# 07 — FCPS evaluation with interactive U*F threshold

Runs the full **TopoSwarm pipeline** (SOM → swarm → U\*F) on a chosen FCPS dataset.
A slider lets you override the automatic U\*F threshold to explore how sensitive
the cluster boundaries are. The map and cluster count update on each release.

**Datasets:** atom, chainlink, hepta, twodiamonds, wingnut, lsun

In [1]:
from pathlib import Path
import subprocess
import tempfile

import numpy as np
import pandas as pd
import altair as alt
import ipywidgets as widgets
from IPython.display import display

from pyesom import ESOM
from pyesom.clustering.ustar_flood import UStarFloodClustering
from pyesom.visualization.cluster import plot_ustar_cluster_overlay

# Locate project root by walking up until pyproject.toml is found.
# Works regardless of whether the kernel CWD is the notebook dir or the project root.
ROOT = Path().resolve()
for _ in range(6):
    if (ROOT / "pyproject.toml").exists():
        break
    ROOT = ROOT.parent
else:
    raise RuntimeError(f"Could not locate project root from {Path().resolve()}")

FCPS_PATH    = ROOT / "tests/fixtures/fcps.npz"
JULIA_PROJ   = ROOT / "toposwarm"
JULIA_SCRIPT = ROOT / "toposwarm/scripts/run_pswarm.jl"
print(f"ROOT: {ROOT}")

z = np.load(FCPS_PATH)
DATASETS = {k.replace("_data", ""): (z[k], z[k.replace("_data", "_cls")])
            for k in z if k.endswith("_data")}
print("Datasets:", list(DATASETS))

ROOT: /Users/mattijnvanhoek/mattijn/pyesom
Datasets: ['atom', 'chainlink', 'hepta', 'twodiamonds', 'wingnut', 'lsun']


In [2]:
# ── pick a dataset ──────────────────────────────────────────────────────────
DATASET = "twodiamonds"   # atom | chainlink | hepta | twodiamonds | wingnut | lsun
N_NODES = 200
EPOCHS  = 30

X, cls = DATASETS[DATASET]
X = X.astype(np.float64)
X = (X - X.mean(0)) / (X.std(0) + 1e-9)
print(f"{DATASET}: {X.shape}  classes: {sorted(set(cls.tolist()))}")

twodiamonds: (800, 2)  classes: [0, 1]


In [3]:
# ── Stage 1: SOM quantization ───────────────────────────────────────────────
_tmp        = Path(tempfile.mkdtemp())
bridge_path = _tmp / "bridge.npz"
result_path = _tmp / "result.npz"

esom = ESOM(n_nodes=N_NODES, backend="intrasom",
            intrasom_kwargs={"mapshape": "toroid", "lattice": "hexa"})
esom.fit(X, epochs=EPOCHS)
esom.export_npz(bridge_path, X, labels=cls)

qe = esom.quantization_error(X)
print(f"Quantization error : {qe:.4f}")

Using fontManager instance from /Users/mattijnvanhoek/.matplotlib/fontlist-v390.json


Loading dataframe...
Normalizing data...
Creating neighborhood...
Initializing map...


Creating Neuron Distance Rows:   0%|          | 0/16 [00:00<?, ?rows/s]

Starting Training...
Rough Training:


  0%|          | 0/15 [00:00<?, ?it/s]

Fine Tuning:


  0%|          | 0/15 [00:00<?, ?it/s]

Training completed successfully.
Quantization error : 0.1148


In [4]:
# ── Stage 2: swarm projection (Julia) ──────────────────────────────────────
proc = subprocess.run(
    ["julia", f"--project={JULIA_PROJ}", str(JULIA_SCRIPT),
     str(bridge_path), str(result_path)],
    capture_output=True, text=True, cwd=ROOT,
)
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr)
    raise RuntimeError("Julia step failed")

r = np.load(result_path)
b = np.load(bridge_path)

sample_rows = r["sample_rows"]
sample_cols = r["sample_cols"]
node_rows   = r["node_rows"]
node_cols   = r["node_cols"]
U           = r["umatrix"]
stress      = float(r["stress"][0])
hit_map     = b["hit_map"]

P = np.zeros_like(U)
for k, (nr, nc) in enumerate(zip(node_rows, node_cols)):
    P[nr - 1, nc - 1] = hit_map[k]

bmu_2d   = np.column_stack([sample_rows - 1, sample_cols - 1])

print(f"Stress : {stress:.6f}")
print(f"Grid   : {U.shape}")

Reading bridge: /var/folders/b3/pctp5dwn6t7fm1t4sc4tknb00000gp/T/tmpuwvw_xn5/bridge.npz
Building distance matrix  (240 nodes × 2 features)
Running TopoSwarm...

Bots  : 240
Grid  : 29 × 29 = 841 cells
Rmax  : 14  Rmin : 1

R=14  epochs= 10  stress=0.0408
R=13  epochs= 10  stress=0.0363
R=12  epochs= 10  stress=0.0332
R=11  epochs= 10  stress=0.0313
R=10  epochs= 10  stress=0.0285
R= 9  epochs= 10  stress=0.0255
R= 8  epochs= 10  stress=0.0227
R= 7  epochs= 10  stress=0.0200
R= 6  epochs= 10  stress=0.0172
R= 5  epochs= 10  stress=0.0145
R= 4  epochs= 10  stress=0.0123
R= 3  epochs= 10  stress=0.0109
R= 2  epochs= 10  stress=0.0094
R= 1  epochs= 10  stress=0.0089

Done. Final stress : 0.0089

── Convergence ─────────────────────────────────────────────────
Stress History  (140 epochs)

0.0815  │•                                                  
        │•                                                  
        │ •                                                 
        │ •••••      

In [ ]:
# ── Stage 3: interactive U*F threshold ─────────────────────────────────────

# fit once to get ustar_ (U* = U combined with P; fixed regardless of threshold)
_clf0          = UStarFloodClustering(min_cluster_size=2, threshold_anchor="upper", toroidal=True)
_clf0.fit(U, P)
ustar_grid     = _clf0.ustar_
auto_threshold = float(_clf0.threshold_)

# cell size: target ~500px on the longer grid axis, clamped to [3, 20]
cell_pixels = float(max(3, min(20, 500 // max(U.shape))))
print(f"Grid {U.shape}  →  cell_pixels={cell_pixels}")

# normalised U* for the histogram — exclude NaN (unoccupied cells)
lo, hi    = float(np.nanmin(ustar_grid)), float(np.nanmax(ustar_grid))
norm_vals = ((ustar_grid - lo) / (hi - lo + 1e-12)).ravel()
norm_vals = norm_vals[~np.isnan(norm_vals)]
hist_df   = pd.DataFrame({"u": norm_vals.tolist()})


def build_charts(threshold: float) -> alt.HConcatChart:
    clf = UStarFloodClustering(min_cluster_size=2, threshold_anchor="upper", toroidal=True)
    clf.fit(U, P, threshold=threshold)
    sample_labels = clf.predict(bmu_2d)
    n_cl       = clf.n_clusters_
    unassigned = int(np.sum(sample_labels == -1))

    overlay = plot_ustar_cluster_overlay(
        ustar_grid,
        clf.labels_,
        title=f"{DATASET}  ·  threshold {threshold:.2f}  ·  "
              f"{n_cl} cluster(s)  ·  {unassigned} unassigned",
        bmu_indices=bmu_2d,
        sample_labels=cls.astype(str),
        cell_pixels=cell_pixels,
    )

    hist = (
        alt.Chart(hist_df)
        .mark_bar(color="#0891b2", opacity=0.7)
        .encode(
            x=alt.X("u:Q", bin=alt.Bin(maxbins=60), title="normalised U* value"),
            y=alt.Y("count()", title="cells"),
        )
        .properties(width=220, height=160, title="U* distribution")
    )
    rule_current = (
        alt.Chart(pd.DataFrame({"t": [threshold], "label": [f"current {threshold:.2f}"]}))
        .mark_rule(color="#ea580c", strokeWidth=2)
        .encode(x="t:Q", tooltip=alt.Tooltip("label:N"))
    )
    rule_auto = (
        alt.Chart(pd.DataFrame({"t": [auto_threshold], "label": [f"auto {auto_threshold:.2f}"]}))
        .mark_rule(color="#adb5bd", strokeWidth=1, strokeDash=[4, 2])
        .encode(x="t:Q", tooltip=alt.Tooltip("label:N"))
    )

    return alt.hconcat(overlay, hist + rule_current + rule_auto).resolve_scale(
        color="independent"
    )


out = widgets.Output()

def on_change(change):
    with out:
        out.clear_output(wait=True)
        display(build_charts(change["new"]))

slider = widgets.FloatSlider(
    value=auto_threshold,
    min=0.01, max=0.99, step=0.01,
    description="threshold",
    continuous_update=False,
    style={"description_width": "80px"},
    layout=widgets.Layout(width="500px"),
)
slider.observe(on_change, names="value")

with out:
    display(build_charts(auto_threshold))

display(slider, out)

Grid (29, 29)  →  cell_pixels=17.0


FloatSlider(value=0.16161616161616163, continuous_update=False, description='threshold', layout=Layout(width='…

Output()